In [2]:
print("Trainer")

Trainer


In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from transformers.data.data_collator import DataCollatorWithPadding 
from datasets import load_dataset
from tqdm import tqdm
import evaluate 
import numpy as np
import torch

In [6]:
# ── TRAINER — TEXT CLASSIFICATION (BERT on IMDb) ──────────────────────────────
# Task         : Binary sentiment classification — positive vs negative review
#                Input  : raw English movie review text (variable length)
#                Output : class label — 0 (negative) or 1 (positive)
#
# Architecture : BERT-base-uncased
#                Backbone  : 12 transformer encoder layers, hidden dim 768
#                            bidirectional — each token attends to ALL other tokens
#                            WordPiece tokenizer — vocab size 30,522
#                Head      : Linear(768 → 2) on top of [CLS] token representation
#                            [CLS] is a special prepended token whose final hidden
#                            state aggregates sentence-level information
#                            dropout(0.1) before the linear layer
#                Loss      : CrossEntropyLoss — standard for classification
#
# Training     : Fine-tune ALL parameters (full fine-tuning, not LoRA)
#                Optimizer : AdamW — Adam with decoupled weight decay
#                Schedule  : linear warmup → linear decay
#                Evaluation: after every epoch, save best checkpoint by accuracy
#                Early stop: patience=2 — stop if accuracy doesn't improve for 2 evals
#
# Data         : IMDb — 25,000 train / 25,000 test, perfectly balanced (50/50 labels)
#                reviews range from ~20 to ~2,500 words
#                truncated to 512 tokens (BERT's maximum sequence length)
#
# Key insight  : BERT was pre-trained on masked language modeling — it already
#                understands English syntax and semantics. Fine-tuning only teaches
#                it to USE that understanding for a specific classification signal.
# ─────────────────────────────────────────────────────────────────────────────

In [7]:
# ── STEP 1 : Load dataset ─────────────────────────────────────────────────────
# load_dataset("imdb") downloads from HuggingFace Hub
# Returns a DatasetDict with two splits:
#   dataset["train"] — 25,000 rows
#   dataset["test"]  — 25,000 rows
# Each row: {"text": "...", "label": 0 or 1}
#
# label mapping: 0 = negative, 1 = positive
# The dataset is already shuffled and balanced — no preprocessing needed here

dataset = load_dataset("imdb")

print("-- Dataset loaded --")
print(f"  Train size   : {dataset['train'].num_rows}")
#   25000
print(f"  Test size    : {dataset['test'].num_rows}")
#   25000
print(f"  Features     : {dataset['train'].features}")
#   {'text': Value(dtype='string'), 'label': ClassLabel(names=['neg','pos'])}
print(f"  Sample text  : {dataset['train'][0]['text'][:80]}...")
#   "I rented I AM CURIOUS-YELLOW from my video store because of all the buzz I had..."
print(f"  Sample label : {dataset['train'][0]['label']}")
#   0  (negative)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

-- Dataset loaded --
  Train size   : 25000
  Test size    : 25000
  Features     : {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
  Sample text  : I rented I AM CURIOUS-YELLOW from my video store because of all the controversy ...
  Sample label : 0


In [8]:
# ── STEP 2 : Load tokenizer ───────────────────────────────────────────────────
# AutoTokenizer.from_pretrained loads the saved vocabulary and tokenizer config
# For bert-base-uncased this is a WordPiece tokenizer:
#   - lowercase everything (uncased)
#   - split unknown words into subword pieces
#     e.g. "unbelievable" → ["un", "##believable"]
#   - special tokens: [CLS]=101, [SEP]=102, [PAD]=0, [UNK]=100, [MASK]=103
#
# Fast tokenizer (Rust-backed) is loaded by default — significantly faster
# than the Python-only slow tokenizer for batch processing

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("\n-- Tokenizer loaded --")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
#   BertTokenizerFast
print(f"  Vocab size      : {tokenizer.vocab_size}")
#   30522
print(f"  [CLS] token id  : {tokenizer.cls_token_id}")
#   101
print(f"  [SEP] token id  : {tokenizer.sep_token_id}")
#   102
print(f"  [PAD] token id  : {tokenizer.pad_token_id}")
#   0

# quick sanity check on a sample text
sample_enc = tokenizer("Anushka loved the film.", return_tensors="pt")
print(f"  Sample input_ids : {sample_enc['input_ids']}")
#   tensor([[ 101, 4083, 2527, 4216, 1996, 2143, 1012,  102]])
#   [CLS] Anushka loved the film . [SEP]




-- Tokenizer loaded --
  Tokenizer class : BertTokenizer
  Vocab size      : 30522
  [CLS] token id  : 101
  [SEP] token id  : 102
  [PAD] token id  : 0
  Sample input_ids : tensor([[  101,  2019, 20668,  2912,  3866,  1996,  2143,  1012,   102]])


In [9]:
# ── STEP 3 : Tokenize the dataset ─────────────────────────────────────────────
# map() applies tokenize_fn to every row in both splits
# batched=True — process 1,000 rows at a time (default batch_size=1000)
#   much faster than row-by-row because the fast tokenizer parallelizes internally
#
# Inside tokenize_fn:
#   truncation=True  → cut sequences longer than max_length (BERT max = 512)
#   max_length=512   → explicit cap; any review > 512 tokens gets truncated
#   padding=False    → do NOT pad here — DataCollatorWithPadding handles it later
#                      dynamic padding (pad each batch to its longest sequence)
#                      is more efficient than padding everything to 512 upfront
#
# remove_columns=["text"] — drop raw text after tokenizing; Trainer doesn't need it
#
# After map(), each row contains:
#   input_ids       : list[int]  — token ids including [CLS] and [SEP]
#   attention_mask  : list[int]  — 1 for real tokens, 0 for padding
#   token_type_ids  : list[int]  — all 0 for single-sentence tasks (no sentence B)
#   label           : int        — unchanged 0 or 1

def tokenize_fn(batch):
    # batch["text"] is a list of strings when batched=True
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,     # let the collator handle per-batch dynamic padding
    )

print("\n-- Tokenizing dataset --")
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",     # shown in tqdm bar
)

# Trainer expects the label column to be named "labels" (plural)
tokenized = tokenized.rename_column("label", "labels")

print(f"  Train columns : {tokenized['train'].column_names}")
#   ['labels', 'input_ids', 'attention_mask', 'token_type_ids']
print(f"  Sample input_ids length : {len(tokenized['train'][0]['input_ids'])}")
#   512  (this particular review was long — truncated to 512)
print(f"  Sample label            : {tokenized['train'][0]['labels']}")
#   0


-- Tokenizing dataset --


Tokenizing:   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/50000 [00:00<?, ? examples/s]

  Train columns : ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
  Sample input_ids length : 363
  Sample label            : 0


In [10]:
# ── STEP 4 : Data collator ────────────────────────────────────────────────────
# DataCollatorWithPadding does dynamic padding at batch assembly time:
#   - takes a list of tokenized examples (each with different sequence length)
#   - pads input_ids, attention_mask, token_type_ids to the longest in the batch
#   - padding value: pad_token_id=0 for input_ids, 0 for masks
#
# Why not pad in Step 3?
#   If we padded to 512 in Step 3, every batch would process 512 tokens even
#   if the longest review in that batch is 150 tokens — 3x wasted compute.
#   Dynamic padding keeps actual computation proportional to real sequence lengths.
#
# return_tensors="pt" → output PyTorch tensors (default for Trainer)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,           # "longest" — pad each batch to its longest sequence
    return_tensors="pt",
)

print("\n-- Data collator created --")
print(f"  Collator type : {type(collator).__name__}")
#   DataCollatorWithPadding

# verify collator behavior on a tiny manual batch
manual_batch = [tokenized["train"][i] for i in range(4)]
collated     = collator(manual_batch)
print(f"  Batch input_ids shape  : {collated['input_ids'].shape}")
#   torch.Size([4, L])  where L = length of longest sequence among these 4 examples
print(f"  Batch attention_mask   : {collated['attention_mask'].shape}")
#   torch.Size([4, L])



-- Data collator created --
  Collator type : DataCollatorWithPadding
  Batch input_ids shape  : torch.Size([4, 363])
  Batch attention_mask   : torch.Size([4, 363])


In [12]:
import logging
from datasets import disable_progress_bar

# 1. Disable progress bars
disable_progress_bar()

# 2. Raise logging level to hide LOAD REPORT / weight‑status messages
logging.getLogger("transformers").setLevel(logging.ERROR)

In [13]:
# ── STEP 5 : Load model ───────────────────────────────────────────────────────
# AutoModelForSequenceClassification loads BERT backbone + classification head
#
# num_labels=2   — binary classification → head is Linear(768 → 2)
# id2label       — maps integer label → string label (shown in model.config)
# label2id       — reverse mapping
#
# from_pretrained downloads:
#   config.json    — architecture params (num_layers=12, hidden_size=768, ...)
#   model weights  — pre-trained BERT weights (no classification head weights yet)
#
# The classification head is randomly initialized — it will be learned during fine-tuning.
# The backbone weights START from BERT's pre-trained values and are updated by fine-tuning.
#
# Warning you will see: "Some weights ... were not initialized from the model checkpoint"
# This is EXPECTED — it refers to the randomly initialized classification head.

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

print("\n-- Model loaded --")
print(f"  Model class       : {type(model).__name__}")
#   BertForSequenceClassification
print(f"  Hidden size       : {model.config.hidden_size}")
#   768
print(f"  Num hidden layers : {model.config.num_hidden_layers}")
#   12
print(f"  Num attention heads: {model.config.num_attention_heads}")
#   12
print(f"  Vocab size        : {model.config.vocab_size}")
#   30522

# count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters    : {total_params:,}")
#   109,483,778  (~109M)
print(f"  Trainable parameters: {trainable_params:,}")
#   109,483,778  — all params trainable in full fine-tuning


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


-- Model loaded --
  Model class       : BertForSequenceClassification
  Hidden size       : 768
  Num hidden layers : 12
  Num attention heads: 12
  Vocab size        : 30522
  Total parameters    : 109,483,778
  Trainable parameters: 109,483,778


In [14]:
# ── STEP 6 : Define compute_metrics ───────────────────────────────────────────
# compute_metrics is called after every evaluation step
# Trainer passes an EvalPrediction namedtuple:
#   eval_pred.predictions — raw logits, shape [num_eval_examples, num_labels]
#   eval_pred.label_ids   — ground truth labels, shape [num_eval_examples]
#
# Logits are raw scores (not probabilities) — argmax gives the predicted class
# We do NOT apply softmax because argmax(logits) == argmax(softmax(logits))
#
# evaluate.load("accuracy") — loads accuracy metric from HuggingFace evaluate lib
# evaluate.load("f1")       — loads F1 metric
#
# f1 average="weighted" — weighted by class support (handles class imbalance)
#   for IMDb (balanced dataset) weighted F1 ≈ macro F1

accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # logits shape: [N, 2]  e.g. [[ 2.1, -0.4], [-1.2,  3.3], ...]
    # argmax along axis=-1 → shape [N]
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy": acc["accuracy"],
        "f1":       f1["f1"],
    }

print("\n-- compute_metrics defined --")
# quick manual test of compute_metrics
dummy_logits = np.array([[2.1, -0.4], [-1.2, 3.3], [0.5, 0.3]])
dummy_labels = np.array([0, 1, 0])
dummy_preds  = np.argmax(dummy_logits, axis=-1)   # [0, 1, 0]
print(f"  Dummy logits  : {dummy_logits.tolist()}")
print(f"  Dummy labels  : {dummy_labels.tolist()}")
print(f"  Predicted     : {dummy_preds.tolist()}")
#   [0, 1, 0]  — all correct in this dummy example



-- compute_metrics defined --
  Dummy logits  : [[2.1, -0.4], [-1.2, 3.3], [0.5, 0.3]]
  Dummy labels  : [0, 1, 0]
  Predicted     : [0, 1, 0]


In [15]:
# ── STEP 7 : TrainingArguments ────────────────────────────────────────────────
# TrainingArguments is a dataclass that controls every aspect of the training loop
#
# output_dir             — where checkpoints and the final model are saved
# num_train_epochs=3     — 3 full passes over 25,000 training examples
# per_device_train_batch_size=16  — 16 examples per GPU per step
#   with 1 GPU: effective batch = 16
#   with 2 GPUs: effective batch = 32 (Trainer handles DDP automatically)
# per_device_eval_batch_size=64   — larger batch for eval (no gradients → less memory)
# learning_rate=2e-5     — standard for BERT fine-tuning (range: 1e-5 to 5e-5)
# weight_decay=0.01      — L2 regularization on all weights except bias and LayerNorm
# warmup_ratio=0.1       — first 10% of steps linearly ramp LR from 0 → 2e-5
#                          then linearly decay from 2e-5 → 0 over remaining steps
# fp16=True              — mixed precision: forward pass in float16, gradients in float32
#                          ~2x speedup on Volta+ GPUs (V100, A100, RTX 3090+)
#                          set to False if using CPU or older GPUs
#
# eval_strategy="epoch"  — evaluate on test set after every epoch
# save_strategy="epoch"  — must match eval_strategy when load_best_model_at_end=True
# load_best_model_at_end=True  — after training, restore the best checkpoint
# metric_for_best_model="accuracy"  — "best" = highest accuracy on eval set
# greater_is_better=True            — accuracy: higher is better
#
# logging_steps=100      — log training loss every 100 steps to console + TensorBoard
# report_to="none"       — disable W&B / other logging integrations for this example

args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,        # keep only 2 checkpoints on disk (deletes older ones)
    seed=42,
    report_to="none",
)

print("\n-- TrainingArguments created --")

# compute expected training steps
steps_per_epoch = len(tokenized["train"]) // args.per_device_train_batch_size
# 25000 // 16 = 1562 steps per epoch (last incomplete batch dropped by default)
warmup_steps    = int(steps_per_epoch * args.num_train_epochs * args.warmup_ratio)
total_steps     = steps_per_epoch * args.num_train_epochs

print(f"  Steps per epoch : {steps_per_epoch}")
#   1562
print(f"  Total steps     : {total_steps}")
#   4686
print(f"  Warmup steps    : {warmup_steps}")
#   468  (first 10% of 4686)


-- TrainingArguments created --
  Steps per epoch : 1562
  Total steps     : 4686
  Warmup steps    : 468


In [16]:
# ── STEP 8 : EarlyStoppingCallback ────────────────────────────────────────────
# Hooks into the training loop via the TrainerCallback interface
# Called at the end of every evaluation (every epoch here)
#
# early_stopping_patience=2  — stop training if accuracy does NOT improve for 2
#                              consecutive evaluations (2 epochs without improvement)
# early_stopping_threshold=0.001  — improvement < 0.001 in accuracy is ignored
#                                   (prevents stopping on noise)
#
# When triggered, sets control.should_training_stop = True
# Trainer checks this flag after each evaluation and stops gracefully

early_stop = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
)

print("\n-- EarlyStoppingCallback created --")
print(f"  Patience  : {early_stop.early_stopping_patience} epochs")
#   2
print(f"  Threshold : {early_stop.early_stopping_threshold}")
#   0.001




-- EarlyStoppingCallback created --
  Patience  : 2 epochs
  Threshold : 0.001


In [18]:
# ── STEP 9 : Trainer ──────────────────────────────────────────────────────────
# Trainer wraps the entire training + evaluation loop
# It handles: batching, forward pass, loss computation, backward pass,
#             optimizer step, LR schedule step, gradient clipping,
#             mixed precision, checkpointing, logging, evaluation, callbacks
#
# model        — the model to train (must be on CPU here; Trainer moves it to GPU)
# args         — TrainingArguments from Step 7
# train_dataset — tokenized["train"] — HuggingFace Dataset, Trainer wraps it in DataLoader
# eval_dataset  — tokenized["test"]  — evaluated after every epoch
# tokenizer     — passed so Trainer can save it alongside the model checkpoint
# data_collator — DataCollatorWithPadding from Step 4 (dynamic padding per batch)
# compute_metrics — function from Step 6 (called after each eval)
# callbacks    — [EarlyStoppingCallback] from Step 8
#
# Internally Trainer creates:
#   DataLoader(train_dataset, batch_size=16, collate_fn=collator, shuffle=True)
#   DataLoader(eval_dataset,  batch_size=64, collate_fn=collator, shuffle=False)
#   AdamW optimizer with weight_decay applied to all params except bias + LayerNorm
#   get_linear_schedule_with_warmup LR scheduler

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stop],
)

print("\n-- Trainer created --")
print(f"  Train batches per epoch : {len(trainer.get_train_dataloader())}")
#   1562
print(f"  Eval batches            : {len(trainer.get_eval_dataloader())}")
#   391  (25000 // 64)


-- Trainer created --
  Train batches per epoch : 1563
  Eval batches            : 391


In [19]:
# ── STEP 10 : Train ───────────────────────────────────────────────────────────
# trainer.train() runs the full training loop:
#
# For each epoch:
#   For each batch (with tqdm progress bar from Trainer):
#     1. Collate: DataCollatorWithPadding pads batch to longest sequence
#     2. Forward: model(input_ids, attention_mask, labels) → loss, logits
#        loss = CrossEntropyLoss(logits, labels) — computed inside model
#     3. Backward: loss.backward() — computes gradients for all 109M parameters
#     4. Clip: torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#     5. Step: optimizer.step() — AdamW update
#     6. Schedule: lr_scheduler.step() — decay LR
#     7. Zero grad: optimizer.zero_grad()
#   End batch loop
#   Evaluate: model.eval() → eval loop → compute_metrics → log accuracy, F1
#   Checkpoint: save if accuracy improved → ./bert-imdb/checkpoint-{step}/
#   EarlyStop: check patience; stop if triggered
#
# After training: load best checkpoint (load_best_model_at_end=True)

print("\n-- Starting training --")
train_result = trainer.train()

print("\n-- Training complete --")
print(f"  Total training time : {train_result.metrics['train_runtime']:.1f}s")
#   ~2800s on a single V100 (16GB)
print(f"  Train samples/sec   : {train_result.metrics['train_samples_per_second']:.1f}")
#   ~27 samples/sec
print(f"  Final train loss    : {train_result.metrics['train_loss']:.4f}")
#   ~0.1352  (converged well — started ~0.69 at epoch 1 step 1)



-- Starting training --


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 

In [ ]:
# ── STEP 11 : Evaluate on test set ────────────────────────────────────────────
# trainer.evaluate() runs a full pass over eval_dataset
# Uses model.eval() — disables dropout for deterministic outputs
# Returns dict with all metrics returned by compute_metrics + eval_loss
#
# eval_loss     — CrossEntropyLoss averaged over all 25,000 test examples
# eval_accuracy — fraction of correct predictions
# eval_f1       — weighted F1 score

print("\n-- Evaluating on test set --")
eval_results = trainer.evaluate()

print(f"  eval_loss     : {eval_results['eval_loss']:.4f}")
#   ~0.1891
print(f"  eval_accuracy : {eval_results['eval_accuracy']:.4f}")
#   ~0.9356  (93.56% accuracy on IMDb test set — competitive result)
print(f"  eval_f1       : {eval_results['eval_f1']:.4f}")
#   ~0.9356  (balanced dataset so accuracy ≈ weighted F1)

In [ ]:
# ── STEP 12 : Manual inference check ─────────────────────────────────────────
# Verify the trained model with a few hand-crafted examples
# Shows the full inference pipeline: text → tokenizer → model → softmax → label
#
# model.eval()     — disable dropout
# torch.no_grad()  — no gradient tracking (not training, save memory + speed)

model.eval()

test_reviews = [
    "Karan absolutely loved the cinematography. A masterpiece of modern cinema.",
    "Harsha walked out after 20 minutes. Possibly the worst film I have ever seen.",
    "Rohit thought it was okay. Some good scenes but overall a bit forgettable.",
]

print("\n-- Manual inference --")
for review in tqdm(test_reviews, desc="Classifying"):
    # tokenize single review
    enc = tokenizer(
        review,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    # forward pass
    with torch.no_grad():
        outputs = model(**enc)
        # outputs.logits shape: [1, 2]  — raw scores for [negative, positive]

    logits      = outputs.logits                        # [1, 2]
    probs       = torch.softmax(logits, dim=-1)         # [1, 2]  sums to 1
    pred_id     = logits.argmax(dim=-1).item()          # 0 or 1
    pred_label  = model.config.id2label[pred_id]        # "negative" or "positive"
    confidence  = probs[0][pred_id].item()              # probability of predicted class

    print(f"\n  Review     : {review[:60]}...")
    print(f"  Logits     : neg={logits[0][0]:.3f}  pos={logits[0][1]:.3f}")
    print(f"  Probs      : neg={probs[0][0]:.3f}  pos={probs[0][1]:.3f}")
    print(f"  Prediction : {pred_label}  ({confidence*100:.1f}% confidence)")

In [ ]:
# ── STEP 13 : Save model ──────────────────────────────────────────────────────
# save_pretrained saves:
#   config.json         — model architecture and label mappings
#   model.safetensors   — model weights (safetensors format, not pickle)
#   tokenizer files     — vocab.txt, tokenizer_config.json, special_tokens_map.json
#
# This is the final best checkpoint (load_best_model_at_end=True restored it in Step 10)
# push_to_hub() commented out — uncomment to share on HuggingFace Hub

trainer.save_model("./bert-imdb-final")
tokenizer.save_pretrained("./bert-imdb-final")

print("\n-- Model saved --")
print("  Saved to: ./bert-imdb-final/")
print("  Files   : config.json, model.safetensors, vocab.txt, tokenizer_config.json")
# trainer.push_to_hub("your-username/bert-imdb-sentiment")  # uncomment to upload


In [ ]:
# ── DRY RUN : IMDb train set, 1 GPU (V100 16GB), epoch 1, first 3 batches ────
#
# Initial state:
#   model   = BERT-base + random Linear(768→2) head
#   dataset = 25,000 tokenized reviews, labels 0/1
#   LR      = 0.0 (warmup starts at 0)
#
# state → action → new state
#
# Batch 1 (global_step=1):
#   state   : LR=0.0, loss=undefined, accuracy=unknown
#   action  : collator pads 16 reviews → input_ids [16, 38]  (longest=38 tokens)
#             forward pass → logits [16,2], loss=0.6931  (ln(2), random init)
#             backward → gradients computed for 109M params
#             optimizer step → all weights nudged by tiny LR=4.3e-8 (warmup step 1)
#             LR incremented: 4.3e-8
#   new state: global_step=1, loss≈0.693, LR=4.3e-8
#
# Batch 100 (global_step=100):
#   state   : LR=4.3e-6 (warmup, step 100/468), loss≈0.55
#   action  : collator pads 16 reviews → input_ids [16, 195]
#             forward → logits [16,2], loss≈0.52
#             backward + step, LR incremented
#   new state: global_step=100, loss≈0.52, LR=4.3e-6
#
# Batch 468 (end of warmup):
#   state   : LR=2e-5 (peak — warmup complete), loss≈0.35
#   action  : forward + backward + step; LR now starts decaying
#   new state: global_step=468, loss≈0.33, LR decaying from 2e-5
#
# Batch 1562 (end of epoch 1):
#   state   : LR≈1.4e-5 (decayed ~30% of way), running loss≈0.25
#   action  : Trainer triggers evaluation
#             full pass over 25,000 test examples in 391 batches of 64
#             compute_metrics called → accuracy≈0.91, f1≈0.91
#             checkpoint saved → ./bert-imdb/checkpoint-1562/
#             EarlyStoppingCallback: best_metric=0.91, patience_counter=0
#   new state: epoch=1 complete, best accuracy=0.91, checkpoint saved
#
# Batch 3124 (end of epoch 2):
#   state   : LR≈0.67e-5 (decaying), loss≈0.15
#   action  : evaluate → accuracy≈0.935
#             checkpoint saved → ./bert-imdb/checkpoint-3124/
#             EarlyStoppingCallback: 0.935 > 0.91 → patience_counter reset to 0
#             checkpoint-1562 deleted (save_total_limit=2 keeps only latest 2)
#   new state: epoch=2 complete, best accuracy=0.935
#
# Batch 4686 (end of epoch 3):
#   state   : LR≈0.0 (fully decayed), loss≈0.11
#   action  : evaluate → accuracy≈0.935  (no improvement over epoch 2)
#             EarlyStoppingCallback: 0.935 - 0.935 < threshold=0.001
#                                    patience_counter → 1  (not 2 yet, continue)
#             But epoch 3 IS the last epoch → training ends naturally
#   new state: training complete
#
# Post-training:
#   load_best_model_at_end=True → loads checkpoint-3124 (accuracy=0.935)
#   trainer.evaluate() → confirms accuracy≈0.9356, f1≈0.9356, loss≈0.189
#   trainer.save_model("./bert-imdb-final") → saves best weights to disk